#  Assignment 1 - Uber Ride Fare Prediction using Machine Learning

**Aim:** Predict the Uber ride fare from pickup and drop-off locations.

This notebook covers all assignment requirements:
1. Data preprocessing
2. Outlier identification and treatment
3. Correlation analysis
4. Linear Regression and Random Forest Regression
5. Model evaluation using R², RMSE and MAE


## 1. Import libraries

In [1]:
import pandas as pd 
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn import metrics


In [3]:
df=pd.read_csv("uber.csv")
df.head()

,Unnamed: 0,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,24238194,2015-05-07 19:52:06.0000003,7.5,2015-05-07 19:52:06 UTC,-73.999817,40.738354,-73.999512,40.723217,1
1,27835199,2009-07-17 20:04:56.0000002,7.7,2009-07-17 20:04:56 UTC,-73.994355,40.728225,-73.994710,40.750325,1
2,44984355,2009-08-24 21:45:00.00000061,12.9,2009-08-24 21:45:00 UTC,-74.005043,40.740770,-73.962565,40.772647,1
3,25894730,2009-06-26 08:22:21.0000001,5.3,2009-06-26 08:22:21 UTC,-73.976124,40.790844,-73.965316,40.803349,3
4,17610152,2014-08-28 17:47:00.000000188,16.0,2014-08-28 17:47:00 UTC,-73.925023,40.744085,-73.973082,40.761247,5


In [4]:
df.isnull().sum()

Unnamed: 0           0
key                  0
fare_amount          0
pickup_datetime      0
pickup_longitude     0
pickup_latitude      0
dropoff_longitude    1
dropoff_latitude     1
passenger_count      0
dtype: int64

In [5]:
#Preproccesed 
df=df.dropna().copy()
df=df[(df["fare_amount"]>0) & (df["passenger_count"]>0)]
df=df[df["dropoff_latitude"].between(-90,90)& df["pickup_latitude"].between(-90,90) & df["dropoff_longitude"].between(-180,180)&
        df["pickup_longitude"].between(-180,180)].copy()

df["pickup_datetime"]=pd.to_datetime(df["pickup_datetime"])
df["hour"]=df["pickup_datetime"].dt.hour

In [5]:
#Haversine formula
lat1=np.radians(df["pickup_latitude"])
lon1=np.radians(df["pickup_longitude"])

lat2=np.radians(df["dropoff_latitude"])
lon2=np.radians(df["dropoff_longitude"])

a=np.sin((lat1-lat2)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon1-lon2)/2)**2
df["distance_km"]=2*6371*np.arcsin(np.sqrt(a))

In [6]:
df.head()

,Unnamed: 0,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,hour,distance_km
0,24238194,2015-05-07 19:52:06.0000003,7.5,2015-05-07 19:52:06+00:00,-73.999817,40.738354,-73.999512,40.723217,1,19,1.683323
1,27835199,2009-07-17 20:04:56.0000002,7.7,2009-07-17 20:04:56+00:00,-73.994355,40.728225,-73.994710,40.750325,1,20,2.457590
2,44984355,2009-08-24 21:45:00.00000061,12.9,2009-08-24 21:45:00+00:00,-74.005043,40.740770,-73.962565,40.772647,1,21,5.036377
3,25894730,2009-06-26 08:22:21.0000001,5.3,2009-06-26 08:22:21+00:00,-73.976124,40.790844,-73.965316,40.803349,3,8,1.661683
4,17610152,2014-08-28 17:47:00.000000188,16.0,2014-08-28 17:47:00+00:00,-73.925023,40.744085,-73.973082,40.761247,5,17,4.475450


In [7]:
features=["passenger_count","hour","distance_km"]
x=df[features]
y=df["fare_amount"]
X_train,X_test,y_train,y_test=train_test_split(x,y,test_size=0.3,random_state=42)

In [8]:
model=LinearRegression()
model.fit(X_train,y_train)
pred=model.predict(X_test)
print("Actual value",y_test.iloc[0])
print("Predicted value",pred[0])

Actual value 11.3
Predicted value 11.430268287691392


In [9]:
print("r2 score is ",metrics.r2_score(y_test,pred))
print("MSE is ", metrics.mean_squared_error(y_test,pred))
print("MAE is :",metrics.mean_absolute_error(y_test,pred))

r2 score is  0.0014789500624389218
MSE is  97.08681829519423
MAE is : 6.038393925426881
